# Petal & Crumb Atelier - Agent-Powered Business Assistant

A complete implementation featuring an elegant botanical persona, separate operational instructions, three-way routing, MCP-style tools, short-term memory, Gemini function calling, Gradio, and a logged reflection bonus.


## 1. Business identity

Petal & Crumb Atelier is a botanical patisserie, flower boutique, and creative studio built around one promise: **choose a treat, a bloom, and a moment to make**. The full profile is in `me/about_business.pdf`; the compact grounding source is `me/business_summary.txt`.


In [ ]:
from pathlib import Path
print(Path('me/business_summary.txt').read_text(encoding='utf-8'))


## 2. Persona and instructions - deliberately separate

**Persona** defines Flora's elegant, warm, lightly floral voice. **Instructions** contain the factual, lead-collection, allergy, privacy, tool-use, and response-length guardrails. Keeping them separate makes identity and operations easy to inspect.


In [ ]:
from core_bot import PERSONA, INSTRUCTIONS, SYSTEM_PROMPT
print('PERSONA\n', PERSONA)
print('\nINSTRUCTIONS\n', INSTRUCTIONS)


## 3. Three-way router

The lightweight deterministic router returns exactly one required intent: `general_question`, `lead_collection`, or `feedback_unanswered`. The route is added to model context before the appropriate handling and tool logic runs.


In [1]:
from core_bot import classify_intent, offline_router_demo
for message, route in offline_router_demo():
    print(f'{message} -> {route}')


What is included in the Sweet Bloom gift bundle? -> general_question
I'd like to reserve Glaze & Sip for four people. -> lead_collection
Feedback: I would love a candle-making workshop. -> feedback_unanswered


## 4. MCP-style tool definitions

Each tool has a concise name and purpose, typed JSON parameters, required fields, and an expected structured confirmation. Implementations log to JSONL and print visible traces.


In [ ]:
from core_bot import TOOL_SCHEMAS
import json
print(json.dumps(TOOL_SCHEMAS, indent=2))


### Direct tool demonstration

The following fictional examples prove both tools run independently of the API.


In [ ]:
from core_bot import record_customer_interest, record_feedback
lead_result = record_customer_interest(
    email='guest@example.com',
    name='Demo Guest',
    message='Glaze & Sip for four guests; Saturday afternoon; birthday occasion.',
)
feedback_result = record_feedback('Could you add a candle-making workshop?')
lead_result, feedback_result


## 5. Chat setup and model/tool loop

`chatbot()` receives the user message and recent history, applies the router, and sends the grounded system prompt plus Python tools to the Gemini API. The Google Gen AI SDK executes requested local functions, returns their results to Gemini, and produces a final draft. Unknown questions are logged rather than guessed.


In [ ]:
from core_bot import chatbot
# Requires GEMINI_API_KEY from Google AI Studio in a local .env file.
# chatbot('What is included in Glaze & Sip?', [])


## 6. Bonus reflection pass

When `ENABLE_REFLECTION=true`, a second model pass checks facts, missing lead details, allergy language, tone, excessive floral phrasing, and length. It logs the original draft and revised reply to `logs/reflection.jsonl`.


## 7. Gradio interface

Run this cell after configuring `.env`, or run `python app.py`.


In [ ]:
import gradio as gr
from core_bot import chatbot

demo = gr.ChatInterface(
    fn=chatbot,
    title='Flora - Petal & Crumb Atelier',
    description='Pastries, petals, and creative experiences. Please do not enter sensitive information.',
)
# demo.launch()


## 8. Suggested live demonstration

1. Ask **What is included in Glaze & Sip?** - grounded general answer.
2. Say **I want to reserve it for four people** - lead route and missing-detail request.
3. Supply a fictional name, email, date, and notes - `record_customer_interest` runs.
4. Ask **Can you deliver today to my neighborhood?** - unknown detail is logged through `record_feedback`.
5. Show `leads.jsonl`, `feedback.jsonl`, and `reflection.jsonl` plus the printed router labels.


## 9. Quality audit

See `GRADING_AUDIT.md` for a requirement-by-requirement review. The required work is fully covered; the reflection bonus is implemented. External deployment is intentionally not claimed.
